<a href="https://colab.research.google.com/github/code-with-bindu/weather-etl-pipeline/blob/main/weather_etl_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import sqlite3
from datetime import datetime

# Using Open-Meteo API - 100% free, no key needed!
cities = {
    "Tokyo": (35.6762, 139.6503),
    "Mumbai": (19.0760, 72.8777),
    "New York": (40.7128, -74.0060),
    "London": (51.5074, -0.1278),
    "Sydney": (-33.8688, 151.2093)
}

print("⏳ Fetching live weather data...")

weather_data = []

for city, (lat, lon) in cities.items():
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true&hourly=relativehumidity_2m"
    response = requests.get(url)
    data = response.json()

    temp = data['current_weather']['temperature']
    wind = data['current_weather']['windspeed']
    humidity = data['hourly']['relativehumidity_2m'][0]

    weather_data.append({
        "city": city,
        "temperature_c": temp,
        "wind_kmph": wind,
        "humidity_%": humidity,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })
    print(f"✅ {city} — {temp}°C | Wind: {wind} km/h | Humidity: {humidity}%")

print("\n🎯 All cities fetched successfully!")

⏳ Fetching live weather data...
✅ Tokyo — 17.8°C | Wind: 6.5 km/h | Humidity: 90%
✅ Mumbai — 26.9°C | Wind: 4.0 km/h | Humidity: 74%
✅ New York — -1.4°C | Wind: 7.0 km/h | Humidity: 51%
✅ London — 8.2°C | Wind: 8.6 km/h | Humidity: 65%
✅ Sydney — 23.7°C | Wind: 8.3 km/h | Humidity: 87%

🎯 All cities fetched successfully!


In [2]:
# Store data in SQLite Database
print("⏳ Storing data in database...")

conn = sqlite3.connect("weather_pipeline.db")
cursor = conn.cursor()

# Create table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS weather_data (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        city TEXT,
        temperature_c REAL,
        wind_kmph REAL,
        humidity_percent REAL,
        timestamp TEXT
    )
""")

# Insert all city data
for row in weather_data:
    cursor.execute("""
        INSERT INTO weather_data (city, temperature_c, wind_kmph, humidity_percent, timestamp)
        VALUES (?, ?, ?, ?, ?)
    """, (row['city'], row['temperature_c'], row['wind_kmph'], row['humidity_%'], row['timestamp']))

conn.commit()
print("✅ Data stored in database successfully!")

# Read back from database to verify
print("\n📊 Reading from Database:")
df_db = cursor.execute("SELECT * FROM weather_data").fetchall()
for record in df_db:
    print(record)

conn.close()
print("\n✅ Pipeline Complete! Data Extracted → Transformed → Loaded into Database!")

⏳ Storing data in database...
✅ Data stored in database successfully!

📊 Reading from Database:
(1, 'Tokyo', 17.8, 6.5, 90.0, '2026-03-19 04:54:39')
(2, 'Mumbai', 26.9, 4.0, 74.0, '2026-03-19 04:54:40')
(3, 'New York', -1.4, 7.0, 51.0, '2026-03-19 04:54:40')
(4, 'London', 8.2, 8.6, 65.0, '2026-03-19 04:54:40')
(5, 'Sydney', 23.7, 8.3, 87.0, '2026-03-19 04:54:41')

✅ Pipeline Complete! Data Extracted → Transformed → Loaded into Database!


In [3]:
import sqlite3

conn = sqlite3.connect("weather_pipeline.db")
cursor = conn.cursor()

print("🌍 WEATHER PIPELINE ANALYSIS REPORT")
print("=" * 40)

# Hottest city
hottest = cursor.execute("SELECT city, temperature_c FROM weather_data ORDER BY temperature_c DESC LIMIT 1").fetchone()
print(f"🔥 Hottest City: {hottest[0]} — {hottest[1]}°C")

# Most humid
humid = cursor.execute("SELECT city, humidity_percent FROM weather_data ORDER BY humidity_percent DESC LIMIT 1").fetchone()
print(f"💧 Most Humid City: {humid[0]} — {humid[1]}%")

# Windiest
windy = cursor.execute("SELECT city, wind_kmph FROM weather_data ORDER BY wind_kmph DESC LIMIT 1").fetchone()
print(f"💨 Windiest City: {windy[0]} — {windy[1]} km/h")

print("=" * 40)
print("✅ Report Generated Successfully!")
conn.close()

🌍 WEATHER PIPELINE ANALYSIS REPORT
🔥 Hottest City: Mumbai — 26.9°C
💧 Most Humid City: Tokyo — 90.0%
💨 Windiest City: London — 8.6 km/h
✅ Report Generated Successfully!
